# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['WRJESWZOMC', 'CANXFBXRIL'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[23, 18, 10,  5, 19, 23, 26, 15, 13,  3],
       [ 3,  1, 14, 24,  6,  2, 24, 18,  9, 12]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  3, 13, 15, 26, 23, 19,  5, 10, 18],
       [ 0, 12,  9, 18, 24,  2,  6, 24, 14,  1]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 3, 13, 15, 26, 23, 19,  5, 10, 18, 23],
       [12,  9, 18, 24,  2,  6, 24, 14,  1,  3]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [11]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64) # batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        # 添加注意力权重矩阵（双线性注意力）
        self.attn_W = tf.keras.layers.Dense(self.hidden, use_bias=False)

        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    # @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        todo
        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # 1. Encoder: 编码输入序列，保存所有时间步的输出用于注意力计算
        enc_emb = self.embed_layer(enc_ids)  # shape: (batch_size, enc_len, 64)
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out: (b_sz, enc_len, hidden)
        
        # 2. Decoder: 解码输出序列
        dec_emb = self.embed_layer(dec_ids)  # shape: (batch_size, dec_len, 64)
        
        # 获取decoder每个时间步的输出
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state)  # dec_out: (b_sz, dec_len, hidden)
        
        # 3. 计算注意力上下文向量（双线性注意力）
        # 对decoder输出做线性变换: (b_sz, dec_len, hidden) -> (b_sz, dec_len, hidden)
        dec_transformed = self.attn_W(dec_out)  # (b_sz, dec_len, hidden)
        
        # 计算注意力分数: (b_sz, dec_len, hidden) 与 (b_sz, enc_len, hidden) 的批量矩阵乘法
        # scores shape: (b_sz, dec_len, enc_len)
        scores = tf.matmul(dec_transformed, enc_out, transpose_b=True)
        
        # 计算注意力权重
        attn_weights = tf.nn.softmax(scores, axis=-1)  # (b_sz, dec_len, enc_len)
        
        # 计算上下文向量
        context = tf.matmul(attn_weights, enc_out)  # (b_sz, dec_len, hidden)
        
        # 4. 结合decoder输出和上下文向量
        combined = tf.concat([dec_out, context], axis=-1)  # (b_sz, dec_len, hidden*2)
        
        # 5. 通过全连接层输出logits
        logits = self.dense(combined)  # (batch_size, dec_len, v_sz)

        return logits
    
    
    # @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
        # 扩展输入维度
        x = tf.expand_dims(x, axis=1)  # shape: (b_sz, 1)
        
        # 1. Embedding
        inp_emb = self.embed_layer(x)  # shape(b_sz, 1, emb_sz)
        
        # 2. RNN Cell 前向传播
        h, new_state = self.decoder_cell(inp_emb[:, 0, :], state)  # h: (b_sz, hidden)
        
        # 3. 计算注意力（双线性注意力）
        # 对当前decoder状态做线性变换: (b_sz, hidden) -> (b_sz, hidden)
        h_transformed = self.attn_W(h)  # (b_sz, hidden)
        
        # 计算注意力分数: (b_sz, 1, hidden) 与 (b_sz, enc_len, hidden) 的批量矩阵乘法
        # scores shape: (b_sz, 1, enc_len)
        scores = tf.matmul(tf.expand_dims(h_transformed, axis=1), enc_out, transpose_b=True)
        scores = tf.squeeze(scores, axis=1)  # (b_sz, enc_len)
        
        # 计算注意力权重
        attn_weights = tf.nn.softmax(scores, axis=-1)  # (b_sz, enc_len)
        
        # 计算上下文向量: (b_sz, hidden)
        context = tf.reduce_sum(tf.expand_dims(attn_weights, axis=-1) * enc_out, axis=1)
        
        # 4. 结合decoder输出和上下文向量
        combined = tf.concat([h, context], axis=-1)  # (b_sz, hidden*2)
        
        # 5. 输出层
        logits = self.dense(combined)  # (b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''
        return out, state

# Loss函数以及训练逻辑

In [12]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [13]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

c:\Users\wangl\.conda\envs\tf_env\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'my_seq2_seq_model_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


step 0 : loss 3.3019035
step 500 : loss 1.3743403
step 1000 : loss 0.4490773
step 1500 : loss 0.190175


<tf.Tensor: shape=(), dtype=float32, numpy=0.1070399284362793>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [14]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]
[('LLLLLLLLLLLLLLLLLLLL', 'KKKWJJWEDBXGFJQADKVL'), ('PPPPPPPPPPPPPPPPPPPP', 'WBAMPQEFVVHIYLMSWMAP'), ('XJXJXJXJXJXJXJXJXJXJ', 'MYKIIGYNFJZPDGFERQWX'), ('HHHHHHHHHHHHHHHHHHHH', 'MHLEQBXNPOTWIZKUBZWH'), ('QQQQQQQQQQQQQQQQQQQQ', 'IDZDGQTSUMRPBMEIDCVW'), ('HHHHHHHHHHHHHHHHHHHH', 'DVHPXAKBTASHRZBNVIFY'), ('TTTTTTTTTTTTTTTTTTTT', 'FKAGYBSLZSARJPJEFVKT'), ('JHYYYYYYYYYYYYYYYYYY', 'QEBXCGBAMBHVABHEEDBJ'), ('RRRRRRRRRRRRRRRRRRRR', 'OJHJZSIEVRWEPKKTTCOR'), ('HHHHHHHHHHHHHHHHHHHH', 'DESYXAZYTRBHPVCUVRUH'), ('SSSSSSSSSSSSSSSSSSSS', 'CDBDUAFLJRHOJQNZDVNS'), ('PPPPPPPPPPPPPPPPPPPP', 'PCGXBUIBWWFMAGFVFCEP'), ('XXXXXXXXXXXXXXXXXXXX', 'QUBMWYEPKVVFPVBHIROX'), ('SSSSSSSSSSSSSSSSSSSS', 'YNQBOXKXOLMGVEWARLFS'), ('IIIIIIIIIIIIIIIIIIII', 'VOFZDBCDSNTDHZUQXNSI'), ('KGKGKGKGKGKGKGKGKGKG',